In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
import os
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from skimage import morphology
from skimage import measure
from skimage.transform import resize
import tensorflow as tf
from sklearn.cluster import KMeans
import matplotlib.patches as patches
import tensorflow.keras.backend as k


2026-01-07 06:22:41.788125: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767766961.802033      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767766961.806201      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-07 06:22:41.824383: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
train_csv=pd.read_csv('../input/osic-pulmonary-fibrosis-progression/train.csv')


In [3]:
train_csv


,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus
0,ID00133637202223847701934,-2,3195,92.856312,83,Male,Never smoked
1,ID00133637202223847701934,2,3203,93.088817,83,Male,Never smoked
2,ID00133637202223847701934,4,3097,90.008138,83,Male,Never smoked
3,ID00133637202223847701934,6,3171,92.158800,83,Male,Never smoked
4,ID00133637202223847701934,8,3115,90.531272,83,Male,Never smoked
...,...,...,...,...,...,...,...
1389,ID00421637202311550012437,29,2716,81.356338,68,Male,Ex-smoker
1390,ID00421637202311550012437,41,2833,84.861011,68,Male,Ex-smoker
1391,ID00421637202311550012437,54,2771,83.003834,68,Male,Ex-smoker
1392,ID00421637202311550012437,70,2628,78.720345,68,Male,Ex-smoker


In [4]:
###########################  BASE WEEK RECORDED #################################
base_week=train_csv.groupby('Patient')['Weeks'].min()
base_week_list=[]
for i in range(len(train_csv)):
    base_week_list.append(base_week[train_csv.iloc[i,0]])
train_csv['base_week']=base_week_list


###########################  COUNT FROM BASE WEEK RECORDED  #################################
base_week=train_csv.groupby('Patient')['Weeks'].min()
count_from_base_week=[]
for i in range(len(train_csv)):
    count_from_base_week.append(train_csv.iloc[i,1]-base_week[train_csv.iloc[i,0]])
train_csv['count_from_base_week']=count_from_base_week

###########################  CONFIDENCE(FOR MODEL)  #################################
confidence=np.zeros(train_csv.shape[0])
train_csv['confidence']=confidence

###########################  BASE FVC  #################################
base_fvc_dict={}
for id in train_csv['Patient'].unique():
    base_fvc_dict[id]=np.array(train_csv[(train_csv['Patient']==id) & (train_csv['Weeks']==base_week[id])]['FVC'])[0]
base_fvc=[]
for i in range(len(train_csv)):
    base_fvc.append(base_fvc_dict[train_csv.iloc[i,0]])
train_csv['base_fvc']=base_fvc

###########################  BASE FEV1  #################################
base_fev1_dict={}
for id in train_csv['Patient'].unique():
    A=train_csv[train_csv['Patient']==id]["base_fvc"].unique()[0]
    B=train_csv[train_csv['Patient']==id]["Age"].unique()[0]
    if train_csv[train_csv['Patient']==id]["Sex"].unique()[0]=='Male':
        base_fev1_dict[id]=0.77*A+0.32+0.0069*B
    else:
        base_fev1_dict[id]=0.77*A+0.28+0.0052*B
        
base_fev1=[]
for i in range(len(train_csv)):
    base_fev1.append(base_fev1_dict[train_csv.iloc[i,0]])
train_csv['base_fev1']=base_fev1

########################## BASE WEEK PERCENT RECORDED ############################### ONLY USE IF NOT USING PERCENT COLUMN ######
base_week_percent_dict={}
for id in train_csv['Patient'].unique():
    base_week_percent_dict[id]=np.array(train_csv[(train_csv['Patient']==id) & (train_csv['Weeks']==base_week[id])]['Percent'])[0]
    
base_week_percent=[]
for i in range(len(train_csv)):
    base_week_percent.append(base_week_percent_dict[train_csv.iloc[i,0]])
train_csv['base_week_percent']=base_week_percent

######################## BASE FEV1/FVC ####################
train_csv['base fev1/base fvc']=train_csv['base_fev1']/train_csv['base_fvc']

####################### BASE HEIGHT ########################
train_csv['base_height']=(train_csv['base_fvc']+9030)/77.0

###################### BASE WEIGHT #######################

base_weight_dict={}
for id in train_csv['Patient'].unique():
    FVC=train_csv[train_csv['Patient']==id]["base_fvc"].unique()[0]
    A=train_csv[train_csv['Patient']==id]["Age"].unique()[0]
    H=train_csv[train_csv['Patient']==id]["base_height"].unique()[0]
    if train_csv[train_csv['Patient']==id]["Sex"].unique()[0]=='Male':
        base_weight_dict[id]=(FVC+5458-49*H+8*A)/12.0
    else:
        base_weight_dict[id]=(FVC+3863-37*H+6*A)/14.0
base_weight=[]
for i in range(len(train_csv)):
    base_weight.append(base_weight_dict[train_csv.iloc[i,0]])
train_csv['base_weight']=base_weight

###################### BASE BMI ########################
train_csv['base_bmi']=train_csv['base_weight']/((train_csv['base_height']/100)**2)


In [5]:
from sklearn.preprocessing import LabelEncoder
lb=LabelEncoder()#sex
train_csv.iloc[:,5]=lb.fit_transform(train_csv.iloc[:,5])
lb2=LabelEncoder()#ss
train_csv.iloc[:,6]=lb2.fit_transform(train_csv.iloc[:,6])


In [6]:
from sklearn.preprocessing import OneHotEncoder
#smoking status
oh1=OneHotEncoder(handle_unknown='ignore')
smoke_cat=pd.DataFrame(oh1.fit_transform(train_csv[['SmokingStatus']]).toarray(),columns=['smoking cat 0','smoking cat 1','smoking cat 2'])
train_csv=pd.concat([train_csv,smoke_cat],axis=1)


In [7]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
train_scaled=pd.DataFrame(sc.fit_transform(train_csv[['Weeks','Age','base_week','count_from_base_week','base_fvc','base_fev1','base_week_percent','base fev1/base fvc','base_height','base_weight','base_bmi']]),columns=['Weeks','Age','base_week','count_from_base_week','base_fvc','base_fev1','base_week_percent','base fev1/base fvc','base_height','base_weight','base_bmi'])
train_scaled['Sex']=train_csv['Sex']
train_scaled['smoking cat 0']=train_csv['smoking cat 0']
train_scaled['smoking cat 1']=train_csv['smoking cat 1']


In [8]:
train_scaled


,Weeks,Age,base_week,count_from_base_week,base_fvc,base_fev1,base_week_percent,base fev1/base fvc,base_height,base_weight,base_bmi,Sex,smoking cat 0,smoking cat 1
0,-1.455549,2.285408,-1.066012,-0.984880,0.552553,0.552754,0.719050,-0.188387,0.552553,0.951469,1.216722,1,0.0,0.0
1,-1.282445,2.285408,-1.066012,-0.757829,0.552553,0.552754,0.719050,-0.188387,0.552553,0.951469,1.216722,1,0.0,0.0
2,-1.195894,2.285408,-1.066012,-0.644303,0.552553,0.552754,0.719050,-0.188387,0.552553,0.951469,1.216722,1,0.0,0.0
3,-1.109342,2.285408,-1.066012,-0.530778,0.552553,0.552754,0.719050,-0.188387,0.552553,0.951469,1.216722,1,0.0,0.0
4,-1.022790,2.285408,-1.066012,-0.417252,0.552553,0.552754,0.719050,-0.188387,0.552553,0.951469,1.216722,1,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1389,-0.113995,0.104508,0.046916,-0.190200,-0.018537,-0.018478,0.140804,-0.080894,-0.018537,0.095565,0.322032,1,0.0,1.0
1390,0.405316,0.104508,0.046916,0.490954,-0.018537,-0.018478,0.140804,-0.080894,-0.018537,0.095565,0.322032,1,0.0,1.0
1391,0.967903,0.104508,0.046916,1.228871,-0.018537,-0.018478,0.140804,-0.080894,-0.018537,0.095565,0.322032,1,0.0,1.0
1392,1.660318,0.104508,0.046916,2.137077,-0.018537,-0.018478,0.140804,-0.080894,-0.018537,0.095565,0.322032,1,0.0,1.0


In [9]:
train_csv


,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,base_week,count_from_base_week,confidence,base_fvc,base_fev1,base_week_percent,base fev1/base fvc,base_height,base_weight,base_bmi,smoking cat 0,smoking cat 1,smoking cat 2
0,ID00133637202223847701934,-2,3195,92.856312,83,1,2,-2,0,0.0,3195,2461.0427,92.856312,0.770279,158.766234,128.121212,50.828203,0.0,0.0,1.0
1,ID00133637202223847701934,2,3203,93.088817,83,1,2,-2,4,0.0,3195,2461.0427,92.856312,0.770279,158.766234,128.121212,50.828203,0.0,0.0,1.0
2,ID00133637202223847701934,4,3097,90.008138,83,1,2,-2,6,0.0,3195,2461.0427,92.856312,0.770279,158.766234,128.121212,50.828203,0.0,0.0,1.0
3,ID00133637202223847701934,6,3171,92.158800,83,1,2,-2,8,0.0,3195,2461.0427,92.856312,0.770279,158.766234,128.121212,50.828203,0.0,0.0,1.0
4,ID00133637202223847701934,8,3115,90.531272,83,1,2,-2,10,0.0,3195,2461.0427,92.856312,0.770279,158.766234,128.121212,50.828203,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1389,ID00421637202311550012437,29,2716,81.356338,68,1,1,15,14,0.0,2739,2109.8192,82.045291,0.770288,152.844156,104.303030,44.647716,0.0,1.0,0.0
1390,ID00421637202311550012437,41,2833,84.861011,68,1,1,15,26,0.0,2739,2109.8192,82.045291,0.770288,152.844156,104.303030,44.647716,0.0,1.0,0.0
1391,ID00421637202311550012437,54,2771,83.003834,68,1,1,15,39,0.0,2739,2109.8192,82.045291,0.770288,152.844156,104.303030,44.647716,0.0,1.0,0.0
1392,ID00421637202311550012437,70,2628,78.720345,68,1,1,15,55,0.0,2739,2109.8192,82.045291,0.770288,152.844156,104.303030,44.647716,0.0,1.0,0.0


In [10]:
sub=pd.read_csv('../input/osic-pulmonary-fibrosis-progression/sample_submission.csv')
test_csv=pd.read_csv('../input/osic-pulmonary-fibrosis-progression/test.csv')


In [11]:
test_week=[]
patient_id=[]
for i in range(len(sub)):
    test_week.append(int(sub.iloc[i,0].split('_')[-1]))
    patient_id.append(sub.iloc[i,0].split('_')[0])
sub['Patient']=patient_id
sub['Weeks']=test_week
sub.drop(['FVC','Confidence'],axis=1,inplace=True)

########## BASE FVC ##########
base_fvc=test_csv.groupby('Patient')['FVC'].min()
fvc=[]
for i in range(len(sub)):
    fvc.append(base_fvc[sub.iloc[i,1]])
sub['base_fvc']=fvc

######### BASE FEV1 ##########
base_fev1_dict_test={}
for id in sub['Patient'].unique():
    A=sub[sub['Patient']==id]["base_fvc"].unique()[0]
    B=test_csv[test_csv['Patient']==id]["Age"].unique()[0]
    if test_csv[test_csv['Patient']==id]["Sex"].unique()[0]=='Male':
        base_fev1_dict_test[id]=0.77*A+0.32+0.0069*B
    else:
        base_fev1_dict_test[id]=0.77*A+0.28+0.0052*B
        
base_fev1_test=[]
for i in range(len(sub)):
    base_fev1_test.append(base_fev1_dict_test[sub.iloc[i,1]])
sub['base_fev1']=base_fev1_test

###############################

sub['base_height']=(sub['base_fvc']+9030)/77.0

##############################


base_weight_dict_test={}
for id in sub['Patient'].unique():
    FVC=sub[sub['Patient']==id]["base_fvc"].unique()[0]
    A=test_csv[test_csv['Patient']==id]["Age"].unique()[0]
    H=sub[sub['Patient']==id]["base_height"].unique()[0]
    if test_csv[test_csv['Patient']==id]["Sex"].unique()[0]=='Male':
        base_weight_dict_test[id]=(FVC+5458-49*H+8*A)/12.0
    else:
        base_weight_dict_test[id]=(FVC+3863-37*H+6*A)/14.0
base_weight_test=[]
for i in range(len(sub)):
    base_weight_test.append(base_weight_dict_test[sub.iloc[i,1]])
sub['base_weight']=base_weight_test

##############################

test_csv.iloc[:,5]=lb.transform(test_csv.iloc[:,5])
test_csv.iloc[:,6]=lb2.transform(test_csv.iloc[:,6])

##############################
percent_dict={}
for id in test_csv['Patient'].unique():
    percent_dict[id]=float(test_csv[test_csv['Patient']==id]['Percent'])
    
sex_dict={}
for id in test_csv['Patient'].unique():
    sex_dict[id]=int(test_csv[test_csv['Patient']==id]['Sex'])

age_dict={}
for id in test_csv['Patient'].unique():
    age_dict[id]=int(test_csv[test_csv['Patient']==id]['Age'])
    
ss_dict={}
for id in test_csv['Patient'].unique():
    ss_dict[id]=int(test_csv[test_csv['Patient']==id]['SmokingStatus'])

percent=[]
sex=[]
age=[]
ss=[]
for i in range(len(sub)):
    percent.append(percent_dict[sub.iloc[i,1]])
    sex.append(sex_dict[sub.iloc[i,1]])
    age.append(age_dict[sub.iloc[i,1]])
    ss.append(ss_dict[sub.iloc[i,1]])    
sub['base_week_percent']=percent
sub['Age']=age
sub['Sex']=sex
sub['SmokingStatus']=ss

###############################

base_week_test=test_csv.groupby('Patient')['Weeks'].min()
count_from_base_week_test=[]
base_week=[]
for i in range(len(sub)):
    count_from_base_week_test.append(sub.iloc[i,2]-base_week_test[sub.iloc[i,1]])
    base_week.append(base_week_test[sub.iloc[i,1]])
sub['count_from_base_week']=count_from_base_week_test
sub['base_week']=base_week

###############################

sub['base fev1/base fvc']=sub['base_fev1']/sub['base_fvc']

###############################
sub['base_bmi']=sub['base_weight']/((sub['base_height']/100.0)**2)


/tmp/ipykernel_11/1348188486.py:61: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  percent_dict[id]=float(test_csv[test_csv['Patient']==id]['Percent'])
/tmp/ipykernel_11/1348188486.py:65: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  sex_dict[id]=int(test_csv[test_csv['Patient']==id]['Sex'])
/tmp/ipykernel_11/1348188486.py:69: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  age_dict[id]=int(test_csv[test_csv['Patient']==id]['Age'])
/tmp/ipykernel_11/1348188486.py:73: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  ss_dict[id]=int(test_csv[test_csv['Patient']==id]['SmokingStatus'])


In [12]:
smoke_cat_test=pd.DataFrame(oh1.transform(sub[['SmokingStatus']]).toarray(),columns=['smoking cat 0','smoking cat 1','smoking cat 2'])
sub=pd.concat([sub,smoke_cat_test],axis=1)


TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [13]:
sub_scaled=pd.DataFrame(sc.transform(sub[['Weeks','Age','base_week','count_from_base_week','base_fvc','base_fev1','base_week_percent','base fev1/base fvc','base_height','base_weight','base_bmi']]),columns=['Weeks','Age','base_week','count_from_base_week','base_fvc','base_fev1','base_week_percent','base fev1/base fvc','base_height','base_weight','base_bmi'])
sub_scaled['Sex']=sub['Sex']
sub_scaled['smoking cat 0']=sub['smoking cat 0']
sub_scaled['smoking cat 1']=sub['smoking cat 1']


KeyError: 'smoking cat 0'

In [14]:
sub_scaled


,Weeks,Age,base_week,count_from_base_week,base_fvc,base_fev1,base_week_percent,base fev1/base fvc,base_height,base_weight,base_bmi,Sex
0,-1.498825,1.558441,0.243315,-2.176901,-0.474407,-0.474214,-1.071756,0.820684,-0.474407,-0.061242,0.453242,1
1,-1.455549,1.558441,0.243315,-2.120138,-0.474407,-0.474214,-1.071756,0.820684,-0.474407,-0.061242,0.453242,1
2,-1.412273,1.558441,0.243315,-2.063375,-0.474407,-0.474214,-1.071756,0.820684,-0.474407,-0.061242,0.453242,1
3,-1.368997,1.558441,0.243315,-2.006612,-0.474407,-0.474214,-1.071756,0.820684,-0.474407,-0.061242,0.453242,1
4,-1.325721,1.558441,0.243315,-1.949849,-0.474407,-0.474214,-1.071756,0.820684,-0.474407,-0.061242,0.453242,1
...,...,...,...,...,...,...,...,...,...,...,...,...
1903,2.872044,0.104508,-0.804146,4.464355,0.700335,0.700361,0.562500,-0.695646,0.700335,0.720615,0.714791,1
1904,2.915320,0.104508,-0.804146,4.521117,0.700335,0.700361,0.562500,-0.695646,0.700335,0.720615,0.714791,1
1905,2.958596,0.104508,-0.804146,4.577880,0.700335,0.700361,0.562500,-0.695646,0.700335,0.720615,0.714791,1
1906,3.001872,0.104508,-0.804146,4.634643,0.700335,0.700361,0.562500,-0.695646,0.700335,0.720615,0.714791,1


In [15]:
#x=np.array(train_csv[['Weeks','Age','Sex','base_week','count_from_base_week','base_fvc','base_fev1','base_week_percent','base fev1/base fvc','base_height','base_weight','base_bmi','smoking cat 0','smoking cat 1']])
x=np.array(train_scaled)
y=np.array(train_csv[['FVC','confidence']])

from sklearn.model_selection import train_test_split
xtrain,xvalid,ytrain,yvalid=train_test_split(x,y,test_size=0.2)


In [16]:
#def quantile_loss(q,ytrue,ypred):
#    eps=1.0
#    sigmas=ypred[:,1]+eps
#    
#    error=ytrue[:,0]-ypred[:,0]
#    return tf.reduce_mean(tf.math.maximum(q*error,(q-1)*error))+tf.math.log(sigmas)


#quantile=[0.1,0.3,0.5,0.7,0.9]
#q_models=[]
#for item in quantile:
#    print('\n\nFOR {} QUANTILE\n\n'.format(item))
#    q_models.append(run_model(xtrain,ytrain,xvalid,yvalid,item))


In [17]:
def metric(actual_fvc, predicted_fvc, confidence, return_values = False):
    """
    Calculates the modified Laplace Log Likelihood score for this competition.
    """
    sd_clipped = np.maximum(confidence, 70)
    delta = np.minimum(np.abs(actual_fvc - predicted_fvc), 1000)
    metric = - np.sqrt(2) * delta / sd_clipped - np.log(np.sqrt(2) * sd_clipped)

    if return_values:
        return metric
    else:
        return np.mean(metric)
    
    


def model_loss(ytrue,ypred):   # this loss penalises both prediction and confidence value
    eps=1.0
    
    fvc_pred=ypred[:,0]
    sigmas=ypred[:,1]+eps     # so as to avoid log(0) . these are predicted connfidences
    
    ans=tf.math.log(sigmas)
    ans=ans+((ytrue[:,0]-fvc_pred)**2)/(2*sigmas**2)
    
    return tf.reduce_mean(ans)


In [18]:
lr_scheduler=tf.keras.callbacks.ReduceLROnPlateau(factor=0.2,monitor='val_loss',mode='min')
class best_weights(tf.keras.callbacks.Callback):
    def __init__(self):
        self.metric_op=-30.0
        self.weights_op=None
        self.epoch_op=-1
    def on_epoch_end(self,epoch,logs={}):
        if logs['val_metric']>=self.metric_op:
            self.metric_op=logs['val_metric']
            self.epoch_op=epoch
            self.weights_op=self.model.get_weights()
    def on_train_end(self,logs={}):
        self.model.set_weights(self.weights_op)
        print('BEST_EPOCH = {}   BEST_SCORE_ON_VALID_SET = {}'.format(self.epoch_op+1,self.metric_op))
        
        

class metrics_call(tf.keras.callbacks.Callback):
    def __init__(self,mertic,xtrain,ytrain,xvalid,yvalid):
        self.metric=metric
        self.xtrain=xtrain
        self.ytrain=ytrain
        self.xvalid=xvalid
        self.yvalid=yvalid
        
    def on_epoch_end(self,epoch,logs={}):
        train_preds=self.model.predict(self.xtrain)
        val_preds=self.model.predict(self.xvalid)
        #print('\r  metric on train set: ',self.metric(self.ytrain[:,0],train_preds[:,0],train_preds[:,1]),end='')
        logs['val_metric']=self.metric(self.yvalid[:,0],val_preds[:,0],val_preds[:,1])
        #print('  metric on valid set: ',self.metric(self.yvalid[:,0],val_preds[:,0],val_preds[:,1]))
        
        

def run_model(xtrain,ytrain,xvalid,yvalid,epoch=50):
    input=tf.keras.layers.Input(shape=xtrain.shape[1:])
    noisy=tf.keras.layers.GaussianNoise(0.3)(input)
    
    d1=tf.keras.layers.Dense(128,activation='relu')(noisy)
    d2=tf.keras.layers.Dense(128,activation='relu')(d1)
    d3=tf.keras.layers.Dense(128,activation='relu')(d2)
    mean_out1=tf.keras.layers.Dense(1)(d3)
    std_den1=tf.keras.layers.Dense(1)(d3)
    
    d4=tf.keras.layers.Dense(128,activation='relu')(noisy)
    d5=tf.keras.layers.Dense(128,activation='relu')(d4)
    d6=tf.keras.layers.Dense(128,activation='relu')(d5)
    mean_out2=tf.keras.layers.Dense(1)(d6)
    std_den2=tf.keras.layers.Dense(1)(d6)
    
    d7=tf.keras.layers.Dense(128,activation='relu')(noisy)
    d8=tf.keras.layers.Dense(128,activation='relu')(d7)
    d9=tf.keras.layers.Dense(128,activation='relu')(d8)
    mean_out3=tf.keras.layers.Dense(1)(d9)
    std_den3=tf.keras.layers.Dense(1)(d9)
    
    mean_combine=tf.keras.layers.Concatenate()([mean_out1,mean_out2,mean_out3])
    std_combine=tf.keras.layers.Concatenate()([std_den1,std_den2,std_den3])
    mean_final=tf.keras.layers.Dense(1)(mean_combine)
    std_final_den=tf.keras.layers.Dense(1)(std_combine)
    std_final=tf.keras.layers.Lambda(lambda x: tf.abs(x))(std_final_den)
    output=tf.keras.layers.Concatenate()([mean_final,std_final])
    model=tf.keras.models.Model(inputs=input,outputs=output)
    
    model.compile(loss=lambda ytrue,ypred: model_loss(ytrue,ypred),optimizer=tf.keras.optimizers.Adam(lr=0.001))
    print(model.summary())
    history=model.fit(xtrain,ytrain,epochs=epoch,batch_size=256,validation_data=(xvalid,yvalid),verbose=0,callbacks=[lr_scheduler,metrics_call(metric,xtrain,ytrain,xvalid,yvalid),best_weights()])
    pd.DataFrame(history.history).plot(figsize=(8, 5))
    plt.ylim(-10,10)
    plt.grid(True)

    return model


In [19]:
model=run_model(xtrain,ytrain,xvalid,yvalid,epoch=1000)


2026-01-07 06:22:49.609495: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1767766969.609972      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:46:00.0, compute capability: 8.6


ValueError: Argument(s) not recognized: {'lr': 0.001}

In [20]:
#xtest=sub[['Weeks','Age','Sex','base_week','count_from_base_week','base_fvc','base_fev1','base_week_percent','base fev1/base fvc','base_height','base_weight','base_bmi','smoking cat 0','smoking cat 1']]
xtest=np.array(sub_scaled)

yans=model.predict(xtest)

sub['FVC']=yans[:,0]
sub['Confidence']=np.abs(yans[:,1])
sub.drop(['Patient','Weeks','Age','Sex','SmokingStatus','base_week','count_from_base_week','base_fvc','base_fev1','base_week_percent','base fev1/base fvc','base_height','base_weight','base_bmi','smoking cat 0','smoking cat 1','smoking cat 2'],axis=1,inplace=True)


NameError: name 'model' is not defined

In [21]:
sub


,Patient_Week,Patient,Weeks,base_fvc,base_fev1,base_height,base_weight,base_week_percent,Age,Sex,SmokingStatus,count_from_base_week,base_week,base fev1/base fvc,base_bmi
0,ID00126637202218610655908_-3,ID00126637202218610655908,-3,2375,1829.6082,148.116883,99.939394,59.375000,78,1,1,-21,18,0.770361,45.554112
1,ID00126637202218610655908_-2,ID00126637202218610655908,-2,2375,1829.6082,148.116883,99.939394,59.375000,78,1,1,-20,18,0.770361,45.554112
2,ID00126637202218610655908_-1,ID00126637202218610655908,-1,2375,1829.6082,148.116883,99.939394,59.375000,78,1,1,-19,18,0.770361,45.554112
3,ID00126637202218610655908_0,ID00126637202218610655908,0,2375,1829.6082,148.116883,99.939394,59.375000,78,1,1,-18,18,0.770361,45.554112
4,ID00126637202218610655908_1,ID00126637202218610655908,1,2375,1829.6082,148.116883,99.939394,59.375000,78,1,1,-17,18,0.770361,45.554112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1903,ID00047637202184938901501_98,ID00047637202184938901501,98,3313,2551.7992,160.298701,121.696970,89.929425,68,1,1,96,2,0.770238,47.360879
1904,ID00047637202184938901501_99,ID00047637202184938901501,99,3313,2551.7992,160.298701,121.696970,89.929425,68,1,1,97,2,0.770238,47.360879
1905,ID00047637202184938901501_100,ID00047637202184938901501,100,3313,2551.7992,160.298701,121.696970,89.929425,68,1,1,98,2,0.770238,47.360879
1906,ID00047637202184938901501_101,ID00047637202184938901501,101,3313,2551.7992,160.298701,121.696970,89.929425,68,1,1,99,2,0.770238,47.360879


In [22]:
sub.to_csv('submission.csv',index=False)
